In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from prophet import Prophet

warnings.filterwarnings('ignore')

print("--- 1. LOADING DATA ---")
data_path = 'data/7_cities_ML_ready.csv'

if not os.path.exists(data_path):
    raise FileNotFoundError(f"File not found at '{data_path}'!")

df = pd.read_csv(data_path)
df['Datetime'] = pd.to_datetime(df['Datetime'])

# Resample to Daily Average
df_daily = df.groupby(['City', pd.Grouper(key='Datetime', freq='D')])['AQI'].mean().reset_index()

# Filter for Delhi and take recent data
city_name = 'Delhi'
city_df = df_daily[df_daily['City'] == city_name][['Datetime', 'AQI']].dropna()
city_df = city_df.sort_values('Datetime').tail(730)  # Last 2 years
city_df.columns = ['ds', 'y']  # Renaming for Prophet

# Set Cap and Floor to prevent unrealistic predictions
city_df['cap'] = 500
city_df['floor'] = 10

# Train (80%) / Test (20%) Split
train_size = int(len(city_df) * 0.8)
train = city_df.iloc[:train_size].copy()
test = city_df.iloc[train_size:].copy()

print(f"City: {city_name} | Train Rows: {len(train)} | Test Rows: {len(test)}")

# --- MODEL 1: Persistence Baseline ---
test['baseline_pred'] = city_df['y'].shift(1).iloc[train_size:].values
test_base = test.dropna(subset=['baseline_pred'])
mae_base = mean_absolute_error(test_base['y'], test_base['baseline_pred'])

# --- MODEL 2: Bounded Prophet Model ---
print("\n--- 2. TRAINING BOUNDED PROPHET MODEL ---")
model_prophet = Prophet(growth='logistic', daily_seasonality=False, yearly_seasonality=True)
model_prophet.fit(train)

future = model_prophet.make_future_dataframe(periods=len(test), freq='D')
future['cap'] = 500
future['floor'] = 10
forecast = model_prophet.predict(future)
test['prophet_pred'] = forecast.iloc[train_size:]['yhat'].values

mae_prophet = mean_absolute_error(test['y'], test['prophet_pred'])

print("\n================ EVALUATION SUMMARY ================")
print(f"Persistence Baseline MAE : {mae_base:.2f}")
print(f"Prophet Model MAE        : {mae_prophet:.2f}")

# --- 3. SAVE MODEL FOR AGENT ---
os.makedirs('models', exist_ok=True)
final_model = Prophet(growth='logistic', daily_seasonality=False, yearly_seasonality=True)
final_model.fit(city_df)

model_file = 'models/prophet_delhi.pkl'
joblib.dump(final_model, model_file)
print(f"\n✓ SUCCESS! Bounded model saved to '{model_file}'!")


      AI ADVISORY AGENT — LOCAL DEMO RUN

✓ AI Agent: Loaded forecasting model successfully.
Forecast Date: 2026-08-07
Predicted AQI: 218

--- Profile: Asthma ---
AQI Band: Poor (Band Indicator: Orange)
Advisory: Warning for sensitive groups! Predicted AQI tomorrow is 218 (Poor). High health risk. Stay indoors, keep windows closed, run air purifiers, and keep emergency inhalers ready.
Translation: [Hindi Regional Text Output]: Warning for sensitive groups! Predicted AQI tomorrow is 218 (Poor). High health risk. Stay indoors, keep windows closed, run air purifiers, and keep emergency inhalers ready.

--- Profile: Outdoor Worker ---
AQI Band: Poor (Band Indicator: Orange)
Advisory: Predicted AQI is 218 (Poor). Standard outdoor precautions apply.
Translation: [Hindi Regional Text Output]: Predicted AQI is 218 (Poor). Standard outdoor precautions apply.

--- Profile: General Public ---
AQI Band: Poor (Band Indicator: Orange)
Advisory: Predicted AQI is 218 (Poor). Air quality is acceptable